# Model Context Protocol (MCP) Fundamnetals

The AI ecosystem is evolving rapidly, with Large Language Models (LLMs) and other AI systems becoming increasingly capable. However, these models are often limited by their training data and lack access to real-time information or specialized tools. This limitation hinders the potential of AI systems to provide truly relevant, accurate, and helpful responses in many scenarios.

This is where Model Context Protocol (MCP) comes in. MCP enables AI models to connect with external data sources, tools, and environments, allowing for the seamless transfer of information and capabilities between AI systems and the broader digital world. This interoperability is crucial for the growth and adoption of truly useful AI applications.

MCP is often described as the “USB-C for AI applications.” Just as USB-C provides a standardized physical and logical interface for connecting various peripherals to computing devices, MCP offers a consistent protocol for linking AI models to external capabilities. 

This standardization benefits the entire ecosystem:

- users enjoy simpler and more consistent experiences across AI applications
- AI application developers gain easy integration with a growing ecosystem of tools and data sources
- tool and data providers need only create a single implementation that works with multiple AI applications
- the broader ecosystem benefits from increased interoperability, innovation, and reduced fragmentation



MCP Solves Fragmentation in AI Interactions

Before MCP, integrating models with tools required:

- Custom code per tool-model pair
- Non-standard APIs for each vendor
- Frequent breaks due to updates
- Poor scalability with more tools


The MCP intoduced in 2024 by Antropic

- Announcement: https://www.anthropic.com/news/model-context-protocol
- Documentation: https://modelcontextprotocol.io/docs/2026-07-28/getting-started/intro
- SDKs: https://github.com/modelcontextprotocol
- Python-SDK: https://github.com/modelcontextprotocol/python-sdk

![](images/9.avif)

The M×N Integration Problem refers to the challenge of connecting M different AI applications to N different external tools or data sources without a standardized approach.

Without a protocol like MCP, developers would need to create M×N custom integrations—one for each possible pairing of an AI application with an external capability.

![](./images/1.png)

![](./images/2.png)

MCP transforms this into an M+N problem by providing a standard interface: each AI application implements the client side of MCP once, and each tool/data source implements the server side once. This dramatically reduces integration complexity and maintenance burden.

![](./images/3.png)

## Example:

```mermaid
---
title: MCP Architecture and Component Interactions
description: A diagram showing the flows of the components in MCP.
---
graph TD
    Client[MCP Client/Application] -->|Sends Request| H[MCP Host]
    H -->|Invokes| A[AI Model]
    A -->|Tool Call Request| H
    H -->|MCP Protocol| T1[MCP Server Tool 01: Web Search]
    H -->|MCP Protocol| T2[MCP Server Tool 02: Calculator tool]
    H -->|MCP Protocol| T3[MCP Server Tool 03: Database Access tool]
    H -->|MCP Protocol| T4[MCP Server Tool 04: File System tool]
    H -->|Sends Response| Client

    subgraph "MCP Host Components"
        H
        G[Tool Registry]
        I[Authentication]
        J[Request Handler]
        K[Response Formatter]
    end

    H <--> G
    H <--> I
    H <--> J
    H <--> K

    style A fill:#f9d5e5,stroke:#333,stroke-width:2px
    style H fill:#eeeeee,stroke:#333,stroke-width:2px
    style Client fill:#d5e8f9,stroke:#333,stroke-width:2px
    style G fill:#fffbe6,stroke:#333,stroke-width:1px
    style I fill:#fffbe6,stroke:#333,stroke-width:1px
    style J fill:#fffbe6,stroke:#333,stroke-width:1px
    style K fill:#fffbe6,stroke:#333,stroke-width:1px
    style T1 fill:#c2f0c2,stroke:#333,stroke-width:1px
    style T2 fill:#c2f0c2,stroke:#333,stroke-width:1px
    style T3 fill:#c2f0c2,stroke:#333,stroke-width:1px
    style T4 fill:#c2f0c2,stroke:#333,stroke-width:1px
```

### Example: Scalable Agent Solution

```mermaid
---
title: Scalable Agent Solution with MCP
description: A diagram illustrating how a user interacts with an LLM that connects to multiple MCP servers, with each server providing both knowledge and tools, creating a scalable AI system architecture
---
graph TD
    User -->|Prompt| LLM
    LLM -->|Response| User
    LLM -->|MCP| ServerA
    LLM -->|MCP| ServerB
    ServerA -->|Universal connector| ServerB
    ServerA --> KnowledgeA
    ServerA --> ToolsA
    ServerB --> KnowledgeB
    ServerB --> ToolsB

    subgraph Server A
        KnowledgeA[Knowledge]
        ToolsA[Tools]
    end

    subgraph Server B
        KnowledgeB[Knowledge]
        ToolsB[Tools]
    end
```

## Some Scenario

```mermaid
---
title: Advanced MCP Scenarios with Client-Server LLM Integration
description: A sequence diagram showing the detailed interaction flow between user, client application, client LLM, multiple MCP servers, and server LLM, illustrating tool discovery, user interaction, direct tool calling, and feature negotiation phases
---
sequenceDiagram
    autonumber
    actor User as 👤 User
    participant ClientApp as 🖥️ Client App
    participant ClientLLM as 🧠 Client LLM
    participant Server1 as 🔧 MCP Server 1
    participant Server2 as 📚 MCP Server 2
    participant ServerLLM as 🤖 Server LLM
    
    %% Discovery Phase
    rect rgb(220, 240, 255)
        Note over ClientApp, Server2: TOOL DISCOVERY PHASE
        ClientApp->>+Server1: Request available tools/resources
        Server1-->>-ClientApp: Return tool list (JSON)
        ClientApp->>+Server2: Request available tools/resources
        Server2-->>-ClientApp: Return tool list (JSON)
        Note right of ClientApp: Store combined tool<br/>catalog locally
    end
    
    %% User Interaction
    rect rgb(255, 240, 220)
        Note over User, ClientLLM: USER INTERACTION PHASE
        User->>+ClientApp: Enter natural language prompt
        ClientApp->>+ClientLLM: Forward prompt + tool catalog
        ClientLLM->>-ClientLLM: Analyze prompt & select tools
    end
    
    %% Scenario A: Direct Tool Calling
    alt Direct Tool Calling
        rect rgb(220, 255, 220)
            Note over ClientApp, Server1: SCENARIO A: DIRECT TOOL CALLING
            ClientLLM->>+ClientApp: Request tool execution
            ClientApp->>+Server1: Execute specific tool
            Server1-->>-ClientApp: Return results
            ClientApp->>+ClientLLM: Process results
            ClientLLM-->>-ClientApp: Generate response
            ClientApp-->>-User: Display final answer
        end
    
    %% Scenario B: Feature Negotiation (VS Code style)
    else Feature Negotiation (VS Code style)
        rect rgb(255, 220, 220)
            Note over ClientApp, ServerLLM: SCENARIO B: FEATURE NEGOTIATION
            ClientLLM->>+ClientApp: Identify needed capabilities
            ClientApp->>+Server2: Negotiate features/capabilities
            Server2->>+ServerLLM: Request additional context
            ServerLLM-->>-Server2: Provide context
            Server2-->>-ClientApp: Return available features
            ClientApp->>+Server2: Call negotiated tools
            Server2-->>-ClientApp: Return results
            ClientApp->>+ClientLLM: Process results
            ClientLLM-->>-ClientApp: Generate response
            ClientApp-->>-User: Display final answer
        end
    end
```

## Core MCP Terminology

### Components

Just like client server relationships in HTTP, MCP has a client and a server.

![](./images/4.png)

- Host: The user-facing AI application that end-users interact with directly. Examples include Anthropic’s Claude Desktop, AI-enhanced IDEs like Cursor, inference libraries like Hugging Face Python SDK, or custom applications built in libraries like LangChain or smolagents. Hosts initiate connections to MCP Servers and orchestrate the overall flow between user requests, LLM processing, and external tools.

- Client: A component within the host application that manages communication with a specific MCP Server. Each Client maintains a 1:1 connection with a single Server, handling the protocol-level details of MCP communication and acting as an intermediary between the Host’s logic and the external Server.

- Server: An external program or service that exposes capabilities (Tools, Resources, Prompts) via the MCP protocol.

### Communication Flow

Let’s examine how these components interact in a typical MCP workflow:

- In the next section, we’ll dive deeper into the communication protocol that enables these components with practical examples.

- User Interaction: The user interacts with the Host application, expressing an intent or query.

- Host Processing: The Host processes the user’s input, potentially using an LLM to understand the request and determine which external capabilities might be needed.

- Client Connection: The Host directs its Client component to connect to the appropriate Server(s).

- Capability Discovery: The Client queries the Server to discover what capabilities (Tools, Resources, Prompts) it offers.

- Capability Invocation: Based on the user’s needs or the LLM’s determination, the Host instructs the Client to invoke specific capabilities from the Server.

- Server Execution: The Server executes the requested functionality and returns results to the Client.

- Result Integration: The Client relays these results back to the Host, which incorporates them into the context for the LLM or presents them directly to the user.

## The Communication Protocol

MCP defines a standardized communication protocol that enables Clients and Servers to exchange messages in a consistent, predictable way. This standardization is critical for interoperability across the community. In this section, we’ll explore the protocol structure and transport mechanisms used in MCP.

### JSON-RPC: The Foundation

At its core, MCP uses JSON-RPC 2.0 as the message format for all communication between Clients and Servers. JSON-RPC is a lightweight remote procedure call protocol encoded in JSON, which makes it:

- Human-readable and easy to debug
- Language-agnostic, supporting implementation in any programming environment
- Well-established, with clear specifications and widespread adoption

The protocol defines three types of messages:

**1. Requests**

Sent from Client to Server to initiate an operation. A Request message includes:

- A unique identifier (id)
- The method name to invoke (e.g., tools/call)
- Parameters for the method (if any)

Example Request:

```json
{
  "jsonrpc": "2.0",
  "id": 1,
  "method": "tools/call",
  "params": {
    "name": "weather",
    "arguments": {
      "location": "San Francisco"
    }
  }
}
```

**2. Responses**

Sent from Server to Client in reply to a Request. A Response message includes:

- The same id as the corresponding Request
- Either a result (for success) or an error (for failure)

Example Success Response:

```json
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "temperature": 62,
    "conditions": "Partly cloudy"
  }
}
```

Example Error Response:

```json
{
  "jsonrpc": "2.0",
  "id": 1,
  "error": {
    "code": -32602,
    "message": "Invalid location parameter"
  }
}
```

**3. Notifications**

One-way messages that don’t require a response. Typically sent from Server to Client to provide updates or notifications about events.

Example Notification:

```json
{
  "jsonrpc": "2.0",
  "method": "progress",
  "params": {
    "message": "Processing data...",
    "percent": 50
  }
}
```

## The Interaction Lifecycle

The MCP protocol defines a structured interaction lifecycle between Clients and Servers:

###  Initialization

The Client connects to the Server and they exchange protocol versions and capabilities, and the Server responds with its supported protocol version and capabilities.

![](images/5.png)

###  Discovery

The Client requests information about available capabilities and the Server responds with a list of available tools.

![](images/6.png)

###  Execution

The Client invokes capabilities based on the Host’s needs.

![](images/7.png)

###  Termination

The connection is gracefully closed when no longer needed and the Server acknowledges the shutdown request.

![](images/8.png)

## MCP Capabilities

MCP Servers expose a variety of capabilities to Clients through the communication protocol. These capabilities fall into four main categories, each with distinct characteristics and use cases. Let’s explore these core primitives that form the foundation of MCP’s functionality.

###  Tools

Tools are executable functions or actions that the AI model can invoke through the MCP protocol.

- Control: Tools are typically model-controlled, meaning that the AI model (LLM) decides when to call them based on the user’s request and context.
- Safety: Due to their ability to perform actions with side effects, tool execution can be dangerous. Therefore, they typically require explicit user approval.
- Use Cases: Sending messages, creating tickets, querying APIs, performing calculations.

Example: A weather tool that fetches current weather data for a given location:

In [12]:
def get_weather(location: str) -> dict:
    """Get the current weather for a specified location."""
    # Connect to weather API and fetch data
    return {
        "temperature": 72,
        "conditions": "Sunny",
        "humidity": 45
    }

### Resources

Resources provide read-only access to data sources, allowing the AI model to retrieve context without executing complex logic.

- Control: Resources are application-controlled, meaning the Host application typically decides when to access them.
- Nature: They are designed for data retrieval with minimal computation, similar to GET endpoints in REST APIs.
- Safety: Since they are read-only, they typically present lower security risks than Tools.
- Use Cases: Accessing file contents, retrieving database records, reading configuration information.

> Note: We moslty use tools instead of resources in use Resources in specifiec static situations.

###  Prompts

Prompts are predefined templates or workflows that guide the interaction between the user, the AI model, and the Server’s capabilities.

- Control: Prompts are user-controlled, often presented as options in the Host application’s UI.
- Purpose: They structure interactions for optimal use of available Tools and Resources.
- Selection: Users typically select a prompt before the AI model begins processing, setting context for the interaction.
- Use Cases: Common workflows, specialized task templates, guided interactions.

An example:

In [13]:
def code_review(code: str, language: str) -> list:
    """Generate a code review for the provided code snippet."""
    return [
        {
            "role": "system",
            "content": f"You are a code reviewer examining {language} code. Provide a detailed review highlighting best practices, potential issues, and suggestions for improvement."
        },
        {
            "role": "user",
            "content": f"Please review this {language} code:\n\n```{language}\n{code}\n```"
        }
    ]

###  Sampling

Sampling allows Servers to request the Client (specifically, the Host application) to perform LLM interactions.

- Control: Sampling is server-initiated but requires Client/Host facilitation.
- Purpose: It enables server-driven agentic behaviors and potentially recursive or multi-step interactions.
- Safety: Like Tools, sampling operations typically require user approval.
- Use Cases: Complex multi-step tasks, autonomous agent workflows, interactive processes.

Example: A Server might request the Client to analyze data it has processed:

Sampling used for multi-agent systems.

In [16]:
def request_sampling(messages, system_prompt=None, include_context="none"):
    """Request LLM sampling from the client."""
    # In a real implementation, this would send a request to the client
    return {
        "role": "assistant",
        "content": "Analysis of the provided data..."
    }

>     This human-in-the-loop design ensures users maintain control over what the LLM sees and generates. When implementing sampling, it’s important to provide clear, well-structured prompts and include relevant context.

## MCP SDK

We leanrn MCP SDK in python in 

```python
from mcp.server.fastmcp import FastMCP

# Create an MCP server
mcp = FastMCP("Weather Service")

# Tool implementation
@mcp.tool()
def get_weather(location: str) -> str:
    """Get the current weather for a specified location."""
    return f"Weather in {location}: Sunny, 72°F"

# Resource implementation
@mcp.resource("weather://{location}")
def weather_resource(location: str) -> str:
    """Provide weather data as a resource."""
    return f"Weather data for {location}: Sunny, 72°F"

# Prompt implementation
@mcp.prompt()
def weather_report(location: str) -> str:
    """Create a weather report prompt."""
    return f"""You are a weather reporter. Weather report for {location}?"""


# Run the server
if __name__ == "__main__":
    mcp.run()
```

> We can not run FastMCP in jupyter lab due to running asyncio prevoiously by jupyter and FastMCP server can not initiate a new in this thread.

| Language   | SDK Repository                                                                                                 | Maintainer / Organization     | Status          |
|------------|----------------------------------------------------------------------------------------------------------------|-------------------------------|-----------------|
| TypeScript | [github.com/modelcontextprotocol/typescript-sdk](https://github.com/modelcontextprotocol/typescript-sdk)       | Anthropic                     | Active          |
| Python     | [github.com/modelcontextprotocol/python-sdk](https://github.com/modelcontextprotocol/python-sdk)               | Anthropic                     | Active          |
| Java       | [github.com/modelcontextprotocol/java-sdk](https://github.com/modelcontextprotocol/java-sdk)                   | Spring AI (VMware)            | Active          |
| Kotlin     | [github.com/modelcontextprotocol/kotlin-sdk](https://github.com/modelcontextprotocol/kotlin-sdk)               | JetBrains                     | Active          |
| C#         | [github.com/modelcontextprotocol/csharp-sdk](https://github.com/modelcontextprotocol/csharp-sdk)               | Microsoft                     | Active (Preview)|
| Swift      | [github.com/modelcontextprotocol/swift-sdk](https://github.com/modelcontextprotocol/swift-sdk)                 | loopwork-ai                   | Active          |
| Rust       | [github.com/modelcontextprotocol/rust-sdk](https://github.com/modelcontextprotocol/rust-sdk)                   | Anthropic/Community           | Active          |
| Dart       | [https://github.com/leehack/mcp_dart](https://github.com/leehack/mcp_dart)                                     | Flutter Community             | Active          |

## MCP Security

Modern MCP implementations require layered security approaches that address both traditional software security and AI-specific threats. The rapidly evolving MCP specification continues to mature its security controls, enabling better integration with enterprise security architectures and established best practices.

### OWASP MCP Top 10 Security Risks

1. **MCP01 – Token Mismanagement & Secret Exposure**
   - **Description:** Improper handling or storage of tokens and secrets can lead to unauthorized access.
   - **Azure Mitigation:** Use Azure Key Vault for secure storage and Managed Identity for workload authentication.

2. **MCP02 – Privilege Escalation via Scope Creep**
   - **Description:** Excessive or overly broad permissions allow attackers or agents to gain elevated privileges.
   - **Azure Mitigation:** Implement fine-grained RBAC and enforce Conditional Access policies.

3. **MCP03 – Tool Poisoning**
   - **Description:** Malicious or tampered tools compromise the integrity of agent operations.
   - **Azure Mitigation:** Enforce strict tool validation and integrity verification before execution.

4. **MCP04 – Software Supply Chain Attacks & Dependency Tampering**
   - **Description:** Compromised third-party libraries or dependencies introduce vulnerabilities.
   - **Azure Mitigation:** Leverage GitHub Advanced Security for dependency scanning and vulnerability alerts.

5. **MCP05 – Command Injection & Execution**
   - **Description:** Unsanitized inputs allow attackers to execute arbitrary commands on the host system.
   - **Azure Mitigation:** Apply rigorous input validation and execute all untrusted code within isolated sandboxes.

6. **MCP06 – Intent Flow Subversion**
   - **Description:** Malicious prompts or inputs manipulate the agent's decision-making or intended workflow.
   - **Azure Mitigation:** Deploy Azure AI Content Safety and Prompt Shields to detect and block harmful intent.

7. **MCP07 – Insufficient Authentication & Authorization**
   - **Description:** Weak or missing identity checks allow unauthorized actors to communicate with agents.
   - **Azure Mitigation:** Use Azure Entra ID for identity management and enforce OAuth 2.1 with PKCE for secure authorization.

8. **MCP08 – Lack of Audit and Telemetry**
   - **Description:** Without proper logging, malicious activities or system failures cannot be detected or investigated.
   - **Azure Mitigation:** Integrate Azure Monitor and Application Insights for comprehensive observability and audit trails.

9. **MCP09 – Shadow MCP Servers**
   - **Description:** Unauthorized or unregistered MCP servers operate outside governance, creating blind spots.
   - **Azure Mitigation:** Use Azure API Center for governance and enforce network isolation to restrict unknown endpoints.

10. **MCP10 – Context Injection & Over-Sharing**
    - **Description:** Agents inadvertently share sensitive context or accept poisoned contextual data from untrusted sources.
    - **Azure Mitigation:** Apply strict data classification policies and enforce minimal data exposure practices.

https://github.com/microsoft/mcp-for-beginners/blob/main/02-Security/README.md